# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

# Five-fold grouped diagnostic: direct 18-way model vs. lab->model hierarchy.
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
texts = train['text'].fillna('').astype(str).tolist()
labels = train['model'].to_numpy()
label_order = sorted(np.unique(labels).tolist())

def get_lab(model):
    if model.startswith('gpt-'): return 'OpenAI'
    if model.startswith('gemini-'): return 'Google'
    if model.startswith('claude-'): return 'Anthropic'
    if model.startswith('qwen-'): return 'Qwen'
    if model.startswith('glm-'): return 'ZAI'
    if model.startswith('kimi-'): return 'Moonshot'
    return 'Other'

labs = np.asarray([get_lab(x) for x in labels])
lab_order = sorted(np.unique(labs).tolist())

def stylometric_features(items):
    rows = []
    function_words = ['the','a','an','and','but','or','if','to','of','in','on','for','with','as','is','are','was','be','that','this','it','I','we','you','can','would','should']
    punctuation = '.,;:!?()[]{}"\'/-'
    for text in items:
        words = re.findall(r"\b[\w']+\b", text.lower())
        lengths = np.asarray([len(w) for w in words], dtype=float)
        n_words, n_chars = max(len(words), 1), max(len(text), 1)
        sentences = max(len(re.findall(r'[.!?]+', text)), 1)
        rows.append([
            np.log1p(len(text)), np.log1p(len(words)), np.log1p(sentences),
            lengths.mean() if len(lengths) else 0., lengths.std() if len(lengths) else 0.,
            len(set(words))/n_words, sum(c.isdigit() for c in text)/n_chars,
            sum(c.isupper() for c in text)/n_chars, sum(c.isspace() for c in text)/n_chars,
            text.count('\\n')/n_chars, text.count('\\n\\n')/n_chars,
            *[text.count(ch)/n_chars for ch in punctuation],
            text.count('#')/n_chars, text.count('*')/n_chars, text.count('```')/n_chars,
            *[words.count(w)/n_words for w in function_words],
        ])
    return np.asarray(rows, dtype=np.float32)

def make_text_features():
    return FeatureUnion([
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1,3), min_df=2, sublinear_tf=True, max_features=220000)),
        ('char', TfidfVectorizer(analyzer='char', ngram_range=(2,6), min_df=2, sublinear_tf=True, max_features=260000)),
    ])

style_matrix = stylometric_features(texts)
all_direct, all_hier = [], []
fold_results = []
for fold, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=5).split(texts, labels, groups=groups), 1):
    tf = make_text_features()
    Xtr_text = tf.fit_transform([texts[i] for i in tr_idx])
    Xva_text = tf.transform([texts[i] for i in va_idx])
    sc = StandardScaler()
    Xtr_style = sc.fit_transform(style_matrix[tr_idx])
    Xva_style = sc.transform(style_matrix[va_idx])
    Xtr = hstack([Xtr_text, csr_matrix(Xtr_style)], format='csr')
    Xva = hstack([Xva_text, csr_matrix(Xva_style)], format='csr')
    ytr, yva = labels[tr_idx], labels[va_idx]
    ltr, lva = labs[tr_idx], labs[va_idx]

    direct = LinearSVC(C=1.0, class_weight='balanced')
    direct.fit(Xtr, ytr)
    pred_direct = direct.predict(Xva)

    lab_model = LinearSVC(C=1.0, class_weight='balanced')
    lab_model.fit(Xtr, ltr)
    pred_lab = lab_model.predict(Xva)
    pred_hier = np.empty(len(va_idx), dtype=object)
    for lab in lab_order:
        train_mask = ltr == lab
        val_mask = pred_lab == lab
        if not np.any(val_mask):
            continue
        local = LinearSVC(C=1.0, class_weight='balanced')
        local.fit(Xtr[train_mask], ytr[train_mask])
        pred_hier[val_mask] = local.predict(Xva[val_mask])

    direct_ba = balanced_accuracy_score(yva, pred_direct)
    hier_ba = balanced_accuracy_score(yva, pred_hier)
    lab_ba = balanced_accuracy_score(lva, pred_lab)
    row = {'fold': fold, 'direct_balanced_accuracy': float(direct_ba), 'hierarchical_balanced_accuracy': float(hier_ba), 'lab_balanced_accuracy': float(lab_ba), 'validation_rows': int(len(va_idx))}
    fold_results.append(row)
    all_direct.extend(zip(yva.tolist(), pred_direct.tolist()))
    all_hier.extend(zip(yva.tolist(), pred_hier.tolist()))
    print(f"Fold {fold}: direct={direct_ba:.4f}, lab={lab_ba:.4f}, hierarchy={hier_ba:.4f}")

cm_direct = confusion_matrix([x[0] for x in all_direct], [x[1] for x in all_direct], labels=label_order)
cm_hier = confusion_matrix([x[0] for x in all_hier], [x[1] for x in all_hier], labels=label_order)
report_direct = classification_report([x[0] for x in all_direct], [x[1] for x in all_direct], labels=label_order, output_dict=True, zero_division=0)
report_hier = classification_report([x[0] for x in all_hier], [x[1] for x in all_hier], labels=label_order, output_dict=True, zero_division=0)

def top_confusions(cm):
    pairs=[]
    for i, actual in enumerate(label_order):
        for j, predicted in enumerate(label_order):
            if i != j and cm[i,j] > 0:
                pairs.append({'actual': actual, 'predicted': predicted, 'count': int(cm[i,j])})
    return sorted(pairs, key=lambda x: x['count'], reverse=True)[:20]

summary = {
    'experiment': 'LLM GMC grouped direct-vs-hierarchical fingerprint diagnostic',
    'split_method': '5-fold GroupKFold by prompt_id',
    'total_rows': int(len(train)), 'model_classes': int(len(label_order)),
    'label_order': label_order, 'lab_order': lab_order,
    'fold_results': fold_results,
    'direct_mean': float(np.mean([r['direct_balanced_accuracy'] for r in fold_results])),
    'direct_std': float(np.std([r['direct_balanced_accuracy'] for r in fold_results], ddof=1)),
    'hierarchical_mean': float(np.mean([r['hierarchical_balanced_accuracy'] for r in fold_results])),
    'hierarchical_std': float(np.std([r['hierarchical_balanced_accuracy'] for r in fold_results], ddof=1)),
    'lab_mean': float(np.mean([r['lab_balanced_accuracy'] for r in fold_results])),
    'direct_per_class_recall': {k: float(v['recall']) for k,v in report_direct.items() if k in label_order},
    'hierarchical_per_class_recall': {k: float(v['recall']) for k,v in report_hier.items() if k in label_order},
    'top_direct_confusions': top_confusions(cm_direct),
    'top_hierarchical_confusions': top_confusions(cm_hier),
}
print('\\nSUMMARY:', json.dumps(summary, indent=2))
Path('/kaggle/working/hierarchy_results.json').write_text(json.dumps(summary, indent=2))
print('Saved:', '/kaggle/working/hierarchy_results.json')


## Direct vs hierarchical fingerprint diagnostic

This run reports five-fold grouped balanced accuracy, lab-level performance, per-class recall, and top confusion pairs.
